# RNG seeding control

This notebook uses `CartPole-v1` to show how mouse-gym exposes Gymnasium seeding.

There are two seeding controls:

- `reset_seed` on `EnvConfig` controls the seed stream for underlying Gymnasium `env.reset(...)` calls inside `step()` (e.g. initial state sampling). This is not a public `reset()` on mouse-gym envs.
- Random action sampling is ordinary Gymnasium behavior: `env.sample_random_input()` samples from the underlying action space. Seed that stream with `env.action_space.seed(seed)` when you need reproducible random actions.

The examples build up one idea at a time. In each section, only one seed is changed so you can see exactly what that seed controls.

In [1]:
import numpy as np

from mouse_gym import EnvConfig, make_env


def first_obs(*, reset_seed: int) -> np.ndarray:
    cfg = EnvConfig(id="CartPole-v1", reset_seed=reset_seed, episodes_per_task=5)
    env = make_env(cfg)
    try:
        return env.step(env.sample_random_input())["observation"]
    finally:
        env.close()

## 1. `reset_seed` controls reset-time randomness

Build two envs with the same `reset_seed` and one with a different seed. The first observation after mouse-gym's internal reset should match when `reset_seed` matches and differ when it does not.

In [2]:
obs_a = first_obs(reset_seed=10)
obs_b = first_obs(reset_seed=10)
obs_c = first_obs(reset_seed=11)

print("reset_seed=10, run A:", obs_a.round(3))
print("reset_seed=10, run B:", obs_b.round(3))
print("reset_seed=11, run C:", obs_c.round(3))
print("A == B:", np.allclose(obs_a, obs_b))
print("A != C:", not np.allclose(obs_a, obs_c))

reset_seed=10, run A: [-0.013  0.023  0.022 -0.029]
reset_seed=10, run B: [-0.013  0.023  0.022 -0.029]
reset_seed=11, run C: [0.01  0.01  0.018 0.021]
A == B: True
A != C: True


## 2. `action_space.seed` controls random action sampling

`reset_seed` does not control `sample_random_input()`. Seed the Gymnasium action space directly when you need reproducible random actions.

In [3]:
def sampled_actions(*, reset_seed: int, action_space_seed: int) -> list[int]:
    cfg = EnvConfig(id="CartPole-v1", reset_seed=reset_seed, episodes_per_task=5)
    env = make_env(cfg)
    try:
        env.action_space.seed(action_space_seed)
        return [int(env.sample_random_input()["action"].item()) for _ in range(8)]
    finally:
        env.close()

same_action_seed = sampled_actions(reset_seed=1, action_space_seed=42)
same_action_seed_other_reset = sampled_actions(reset_seed=99, action_space_seed=42)
different_action_seed = sampled_actions(reset_seed=1, action_space_seed=43)

print("action_space.seed=42:", same_action_seed)
print("same action seed, different reset_seed:", same_action_seed_other_reset)
print("different action seed:", different_action_seed)
print("action seed dominates ->", same_action_seed == same_action_seed_other_reset)
print("different action seed ->", same_action_seed != different_action_seed)

action_space.seed=42: [0, 1, 1, 0, 0, 1, 0, 1]
same action seed, different reset_seed: [0, 1, 1, 0, 0, 1, 0, 1]
different action seed: [1, 1, 0, 0, 1, 0, 0, 1]
action seed dominates -> True
different action seed -> True
